<a href="https://colab.research.google.com/github/rajesh-doma/finbot/blob/main/notebooks/01_document_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Document Parsing
Parses all documents from the FinBot data folder using Docling (PDF, DOCX, MD)
and Pandas (CSV). Validates structure before chunking in Notebook 01.

In [1]:
import os

if not os.path.exists('/content/finbot'):
    !git clone --no-checkout --depth 1 https://github.com/rajesh-doma/finbot.git /content/finbot
    %cd /content/finbot
    !git sparse-checkout init --cone
    !git sparse-checkout set data
    !git checkout main
else:
    %cd /content/finbot
    !git pull

Cloning into '/content/finbot'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 35 (delta 3), reused 31 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.09 MiB | 21.89 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/finbot
Already on 'main'
Your branch is up to date with 'origin/main'.


In [36]:
!find /content/finbot/data -type f | sort

/content/finbot/data/engineering/engineering_master_doc.md
/content/finbot/data/engineering/.gitkeep
/content/finbot/data/engineering/incident_report_log.md
/content/finbot/data/engineering/sprint_metrics_2024.md
/content/finbot/data/engineering/system_sla_report_2024.md
/content/finbot/data/finance/department_budget_2024.docx
/content/finbot/data/finance/financial_summary.docx
/content/finbot/data/finance/.gitkeep
/content/finbot/data/finance/quarterly_financial_report.docx
/content/finbot/data/finance/vendor_payments_summary.docx
/content/finbot/data/general/employee_handbook.pdf
/content/finbot/data/general/.gitkeep
/content/finbot/data/hr/.gitkeep
/content/finbot/data/hr/hr_data.csv
/content/finbot/data/marketing/campaign_performance_data.docx
/content/finbot/data/marketing/customer_acquisition_report.docx
/content/finbot/data/marketing/.gitkeep
/content/finbot/data/marketing/marketing_report_2024.docx
/content/finbot/data/marketing/marketing_report_q1_2024.docx
/content/finbot/dat

In [3]:
!pip install -q qdrant-client sentence-transformers docling groq docling-hierarchical-pdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 446.3/446.3 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.1 MB/s eta 0:00:00
  

In [5]:
import hashlib

COLLECTION_ROLES = {
    "general":     ["employee", "finance", "engineering", "marketing", "hr", "c_level"],
    "finance":     ["finance", "c_level"],
    "engineering": ["engineering", "c_level"],
    "marketing":   ["marketing", "c_level"],
    "hr":          ["hr", "c_level"],
}

def _get_page(item):
    try:
        return item.prov[0].page_no if item.prov else 0
    except:
        return 0

def parse_document_flat(doc, source_file, collection):
    access_roles = COLLECTION_ROLES.get(collection, [])
    chunks = []

    # Find minimum heading level to normalize
    all_levels = []
    for item, level in doc.iterate_items():
        if type(item).__name__ == "SectionHeaderItem":
            all_levels.append(level)

    min_level = min(all_levels) if all_levels else 1

    current_section = source_file
    current_page    = 0
    buffer          = []

    def flush(section, page, buf, chunk_type="text"):
        if not buf:
            return
        content = "\n".join(buf).strip()
        if not content:
            return
        chunk_id = hashlib.md5(
            f"{source_file}::{section}::{content[:50]}".encode()
        ).hexdigest()
        chunks.append({
            "chunk_id":        chunk_id,
            "text":            f"[{section}]\n{content}",
            "source_document": source_file,
            "collection":      collection,
            "access_roles":    access_roles,
            "section_title":   section,
            "page_number":     page,
            "chunk_type":      chunk_type,
            "parent_chunk_id": None,
        })
        buf.clear()

    for item, level in doc.iterate_items():
        item_type = type(item).__name__

        if item_type == "SectionHeaderItem":
            flush(current_section, current_page, buffer)
            current_section = item.text.strip()
            current_page    = _get_page(item)

        elif item_type == "TextItem":
            text = item.text.strip()
            if text:
                if sum(len(t) for t in buffer) > 1200:
                    flush(current_section, current_page, buffer)
                buffer.append(text)
                current_page = _get_page(item) or current_page

        elif item_type == "TableItem":
            flush(current_section, current_page, buffer)
            try:
                table_md = item.export_to_markdown()
            except:
                table_md = str(item)
            chunk_id = hashlib.md5(
                f"{source_file}::table::{current_section}".encode()
            ).hexdigest()
            chunks.append({
                "chunk_id":        chunk_id,
                "text":            f"[{current_section}]\n{table_md}",
                "source_document": source_file,
                "collection":      collection,
                "access_roles":    access_roles,
                "section_title":   current_section,
                "page_number":     current_page,
                "chunk_type":      "table",
                "parent_chunk_id": None,
            })

    flush(current_section, current_page, buffer)
    return chunks

print("parse_document_flat defined ✅")

parse_document_flat defined ✅


In [6]:
import os
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

pipeline_opts = PdfPipelineOptions()
pipeline_opts.do_ocr = False
pipeline_opts.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts)
    }
)

test_files = [
    ("/content/finbot/data/finance/department_budget_2024.docx",      "finance"),
    ("/content/finbot/data/engineering/engineering_master_doc.md",     "engineering"),
    ("/content/finbot/data/hr/hr_data.csv",                           "hr"),
    ("/content/finbot/data/marketing/campaign_performance_data.docx",  "marketing"),
]

results = {}

for filepath, collection in test_files:
    print(f"\n{'='*50}")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Collection: {collection}")

    try:
        if filepath.endswith(".csv"):
            import pandas as pd
            df = pd.read_csv(filepath)
            print(f"Rows: {len(df)}  Columns: {list(df.columns)}")
            print(f"Chunks: will be handled separately")
            results[filepath] = "✅ OK (CSV)"
            continue

        result  = converter.convert(filepath)
        doc_tmp = result.document

        headings = sum(1 for item, _ in doc_tmp.iterate_items()
                      if type(item).__name__ == "SectionHeaderItem")
        texts    = sum(1 for item, _ in doc_tmp.iterate_items()
                      if type(item).__name__ == "TextItem")
        tables   = sum(1 for item, _ in doc_tmp.iterate_items()
                      if type(item).__name__ == "TableItem")

        chunks_tmp = parse_document_flat(doc_tmp, os.path.basename(filepath), collection)

        results[filepath] = "✅ OK"
        print(f"Headings: {headings}  Texts: {texts}  Tables: {tables}")
        print(f"Chunks  : {len(chunks_tmp)}")

    except Exception as e:
        results[filepath] = f"❌ ERROR: {e}"
        print(f"ERROR: {e}")

print(f"\n{'='*50}")
print("SUMMARY:")
for f, status in results.items():
    print(f"  {os.path.basename(f):45s} → {status}")


File: department_budget_2024.docx
Collection: finance


Headings: 26  Texts: 324  Tables: 15
Chunks  : 38

File: engineering_master_doc.md
Collection: engineering


Headings: 146  Texts: 238  Tables: 16
Chunks  : 61

File: hr_data.csv
Collection: hr
Rows: 500  Columns: ['employee_id', 'full_name', 'gender', 'role', 'department', 'designation_level', 'email', 'phone', 'location', 'date_of_birth', 'date_of_joining', 'employment_type', 'employment_status', 'manager_id', 'salary', 'leave_balance', 'leaves_taken', 'attendance_pct', 'performance_rating', 'last_review_date', 'exit_date']
Chunks: will be handled separately

File: campaign_performance_data.docx
Collection: marketing


Headings: 23  Texts: 91  Tables: 6
Chunks  : 20

SUMMARY:
  department_budget_2024.docx                   → ✅ OK
  engineering_master_doc.md                     → ✅ OK
  hr_data.csv                                   → ✅ OK (CSV)
  campaign_performance_data.docx                → ✅ OK
